In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TFEQ(nn.Module):

    def __init__(self, channel=3, dmodel=64, nhead=8,
                 dim_feedforward=32, num_layers=1, time_in=400):

        super().__init__()

        self.input_embedding = nn.Linear(channel, dmodel)

        self.temporal_encoder_layer = nn.TransformerEncoderLayer(
            d_model=dmodel,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            batch_first=True      # Important
        )

        self.temporal_encoder = nn.TransformerEncoder(
            self.temporal_encoder_layer,
            num_layers=num_layers
        )

        self.decoder = nn.Linear(dmodel, 1)

        # CPU/GPU compatible
        self.register_buffer(
            "temporal_pos",
            torch.arange(time_in, dtype=torch.long)
        )

        self.temporal_pe = nn.Embedding(time_in, dmodel)

        self.flatten = nn.Flatten()

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(time_in, 2)

    def forward(self, x):

        b, t, c = x.shape

        x = self.input_embedding(x)

        t_pe = self.temporal_pe(self.temporal_pos).unsqueeze(0).expand(b, -1, -1)

        x = x + t_pe

        x = self.temporal_encoder(x)

        x = self.decoder(x)

        x = self.flatten(x)

        x = self.dropout(x)

        x = self.fc(x)

        return F.log_softmax(x, dim=1)

In [2]:
import sys
print(sys.executable)

c:\Users\Trainee_Harshit\TFEQ-for-EQ_detection\venv\Scripts\python.exe


In [ ]:
# utils.py

In [39]:
import torch
import torch.nn.functional as F
import numpy as np
import random
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, classification_report, roc_auc_score,recall_score,precision_score


def train(model, train_loader, optimizer, epoch):
    model.train()
    loss = []
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data, target
        optimizer.zero_grad()
        output = model(data)
#         loss += F.cross_entropy( scores, captions )
        l = F.nll_loss(output, target.squeeze())
        l.backward()
        optimizer.step()
        loss.append(l.item())
    loss = np.mean(loss)
    print('Train Epoch: {}, Average Loss: {:.4f}'.format(epoch, loss))
    return loss


def test(model, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data, target
            output = model(data)
            test_loss += F.nll_loss(output, target.squeeze(),
                                    reduction='sum').item()
            pred = output.max(1, keepdim=True)[1]  #
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(test_loader.dataset)
    print('Test set: Average loss: {:.4f}, Accuracy: {}/{} ({:.2f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))
    return 100. * correct / len(test_loader.dataset)


def summary(target, y_pred):
    print("confusion matrix:\n", confusion_matrix(target, y_pred))
    print('Roc score: %.4f' % roc_auc_score(target, y_pred))
    print("F1 score: %.4f" % f1_score(target, y_pred))
    print('Recall_score: %.4f' % recall_score(target, y_pred))
    print("Precision_score: %.4f" % precision_score(target, y_pred))
    print(classification_report(target, y_pred))


def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True


def enable_dropout(model):
    """ Function to enable the dropout layers during test-time """
    for m in model.modules():
        if m.__class__.__name__.startswith('Dropout'):
            m.train()


In [ ]:
# TRAINING

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os,time,pickle
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [16]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [17]:
from joblib import dump, load

In [40]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

# --------------------------------------------
# Load Dataset
# --------------------------------------------

train_data = np.load("./data_4/data_train.npy", allow_pickle=True).item()
test_data = np.load("./data_4/data_test.npy", allow_pickle=True).item()

x_train = train_data["x_train"]
y_train = train_data["y_train"]

x_test = test_data["x_test"]
y_test = test_data["y_test"]

print("Original Shape")
print(x_train.shape)
print(x_test.shape)

# --------------------------------------------
# Convert (N,3,400) -> (N,400,3)
# --------------------------------------------

x_train = np.transpose(x_train, (0, 2, 1))
x_test = np.transpose(x_test, (0, 2, 1))

print("\nAfter Transpose")
print(x_train.shape)
print(x_test.shape)

# --------------------------------------------
# Convert to Torch
# --------------------------------------------

x_train = torch.from_numpy(x_train).float()
y_train = torch.from_numpy(y_train).long()

x_test = torch.from_numpy(x_test).float()
y_test = torch.from_numpy(y_test).long()

# --------------------------------------------
# DataLoader
# --------------------------------------------

train_data = TensorDataset(x_train, y_train)
test_data = TensorDataset(x_test, y_test)

train_loader = DataLoader(
    train_data,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_data,
    batch_size=256,
    shuffle=False
)

print("\nTrain Loader Ready")

Original Shape
(60848, 3, 400)
(15212, 3, 400)

After Transpose
(60848, 400, 3)
(15212, 400, 3)

Train Loader Ready


In [41]:
import time
import torch
import torch.optim as optim



# ------------------------------------------------
# Seed
# ------------------------------------------------

setup_seed(42)

# ------------------------------------------------
# Model
# ------------------------------------------------

model = TFEQ(channel=3, time_in=400)

# ------------------------------------------------
# Optimizer
# ------------------------------------------------

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# ------------------------------------------------
# Training
# ------------------------------------------------

num_epochs = 5      # Change to 200 after verifying everything works

L = []
test_hist = []

for epoch in range(1, num_epochs + 1):

    print("=" * 60)
    print(f"Epoch {epoch}/{num_epochs}")

    start = time.time()

    loss = train(
        model,
        train_loader,
        optimizer,
        epoch
    )

    L.append(loss)

    print(f"Training Time : {time.time()-start:.2f} sec")

    acc = test(
        model,
        test_loader
    )

    test_hist.append(acc)

print("\nTraining Finished Successfully.")

Epoch 1/5
Train Epoch: 1, Average Loss: 0.3750
Training Time : 881.65 sec
Test set: Average loss: 0.2576, Accuracy: 13646/15212 (89.71%)

Epoch 2/5
Train Epoch: 2, Average Loss: 0.2610
Training Time : 880.57 sec
Test set: Average loss: 0.2296, Accuracy: 13856/15212 (91.09%)

Epoch 3/5
Train Epoch: 3, Average Loss: 0.2397
Training Time : 864.03 sec
Test set: Average loss: 0.2283, Accuracy: 13864/15212 (91.14%)

Epoch 4/5
Train Epoch: 4, Average Loss: 0.2251
Training Time : 882.96 sec
Test set: Average loss: 0.2227, Accuracy: 13930/15212 (91.57%)

Epoch 5/5
Train Epoch: 5, Average Loss: 0.2137
Training Time : 872.27 sec
Test set: Average loss: 0.2085, Accuracy: 13926/15212 (91.55%)


Training Finished Successfully.


In [42]:
model.eval()
y_pred = []
out=[]
y_test = []
with torch.no_grad():
    for data, target in test_loader:
        output = model(data)
        output=F.softmax(output, dim=1)
        pred = output.max(1, keepdim=True)[1]  
        pred = pred.cpu().numpy().squeeze()
        y_pred.append(pred)
        out.append(output.cpu().numpy())
        y_test.append(target.numpy().squeeze())
y_pred=np.hstack(y_pred)
y_test=np.hstack(y_test)
out=np.vstack(out)

summary(y_pred,y_test)

confusion matrix:
 [[7190  841]
 [ 445 6736]]
Roc score: 0.9167
F1 score: 0.9129
Recall_score: 0.9380
Precision_score: 0.8890
              precision    recall  f1-score   support

           0       0.94      0.90      0.92      8031
           1       0.89      0.94      0.91      7181

    accuracy                           0.92     15212
   macro avg       0.92      0.92      0.92     15212
weighted avg       0.92      0.92      0.92     15212



In [43]:
torch.save(model, './harshit_models/TFEQ.pth')